In [1]:
#install needed libraries
#requests pour envoyer des requêtes HTTP vers les sites web
#beautifulsoup4	pour xtraire les informations du HTML
# pandas pour organiser les textes dans un DataFrame
!pip install -q beautifulsoup4 requests pandas

In [2]:
import requests
import pandas as pd
import re
import time

from bs4 import BeautifulSoup

print("Toutes les bibliothèques sont prêtes !")

Toutes les bibliothèques sont prêtes !


In [3]:
columns = ["text", "score", "label", "source"]

df = pd.DataFrame(columns=columns)

print(df.head())
print(df.shape)

Empty DataFrame
Columns: [text, score, label, source]
Index: []
(0, 4)


In [4]:
url = "https://mawdoo3.com/%D9%85%D8%A7_%D9%87%D9%8A_%D8%A7%D9%84%D8%A8%D9%8A%D8%A6%D8%A9"

response = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=20
)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("HTML length:", len(response.text))

Status code: 200
Content-Type: text/html; charset=UTF-8
HTML length: 215911


In [5]:
print(response.text[:1000])

<!DOCTYPE html><html lang="ar" class="client-nojs" dir="rtl"><head><!-- Google tag (gtag.js) --><script async src="https://www.googletagmanager.com/gtag/js?id=G-HK45635168"></script><script>window.dataLayer = window.dataLayer || [];function gtag(){dataLayer.push(arguments);}gtag('js', new Date());gtag('config', 'G-HK45635168',{'content_group': 'تعليم/تعليم/علوم'});</script><link rel="dns-prefetch" href="https://modo3.com" /><meta charset="UTF-8"><meta name="keywords" content="تعليم,علوم,ما,هي,البيئة" ><meta name="description" content=". تعريف البيئة لغة . تعريف البيئة اصطلاحاً . البيئة في علم الاجتماع . البيئة وفق دائرة المعارف الجغرافية الطبيعية . البيئة وفق العالم إرنست هيكل . البيئة" ><meta name="viewport" content="width=device-width, initial-scale=1"><title>ما هي البيئة - موضوع</title><link rel="shortcut icon" href="https://modo3.com/favicon.ico"><link rel="apple-touch-icon" sizes="180x180" href="https://modo3.com/apple-touch-icon-180x180.png"><link href="https://modo3.com/icon-hir

In [6]:
url = "https://mawdoo3.com/ما_هي_البيئة"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
#transforme le HTML en structure navigable
soup = BeautifulSoup(response.text, "html.parser")

# Extraire le titre
title = soup.title.get_text(strip=True) if soup.title else "No title"

print("Article title:", title)

Article title: ما هي البيئة - موضوع


In [7]:
paragraphs = soup.find_all("p")

texts = []

for p in paragraphs:
    text = p.get_text(" ", strip=True)

    # Garder les paragraphes suffisamment longs
    if len(text) > 50:
        texts.append(text)

print("Nombre de paragraphes:", len(texts))

for i, text in enumerate(texts[:5]):
    print(f"\nParagraph {i+1}:")
    print(text)

Nombre de paragraphes: 12

Paragraph 1:
تُعرَف البيئة لغةً بأنّها مجموعة العناصر الاصطناعية والطبيعية التي تُحِيطُ بالكائنات الحية (الإنسان والحيوان والنبات)، وهي أيضاً المنزل والمكان الذي يسكن فيه الكائن الحي، وبشكل عام هي جميع الظروف الخارجية التي تؤثر في شيء معين. [١]

Paragraph 2:
عوامل البيئة هي عبارة عن العوامل التي تحيط وتتحكم في أحوال الكائنات الحية، وكيفية توزيعها، وأماكن عيشها، ويُمكن القول إنّ البيئة هي الحال؛ ويُقال أنّ هناك أنواعاً للبيئة، كالبيئة الطبيعية، والبيئة السياسية، والبيئة الاجتماعية. [١]

Paragraph 3:
ويُشار إلى أنّ كلمة البيئة باللغة الإنجليزية بمصطلح (Environment)، واشتُقّت من الكلمة الفرنسية (Environ)، وتعني المحيط بما يحتويه من عوامل حيوية أو عوامل أحيائية (بالإنجليزية: Biotic Factors)، كالإنسان، والنباتات، والحيوانات، والميكروبات، وغيرها، وعوامل لا حيوية أو عوامل لاأحيائية (بالإنجليزية: Abiotic Factors)، كالماء، والهواء، والتربة، وغيرها. [٢]

Paragraph 4:
تعددت معاني البيئة وتباينت مفاهيمها، وفيما يأتي توضيح لذلك:

Paragraph 5:
البيئة في مفهومها العام تعني 

In [8]:
import re

def clean_arabic_text(text):

    # 1. Remove references such as [١], [2]
    text = re.sub(r'\[[\d٠-٩]+\]', '', text)

    # 2. Remove Arabic diacritics (التشكيل)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)

    # 3. Remove Tatweel (ـ)
    text = text.replace('ـ', '')

    # 4. Normalize spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [9]:
cleaned_texts = []

for text in texts:
    cleaned = clean_arabic_text(text)

    if len(cleaned) > 50:
        cleaned_texts.append(cleaned)

print("Paragraphes avant nettoyage:", len(texts))
print("Paragraphes après nettoyage:", len(cleaned_texts))

print("\nExemple avant:")
print(texts[0])

print("\nExemple après:")
print(cleaned_texts[0])

Paragraphes avant nettoyage: 12
Paragraphes après nettoyage: 10

Exemple avant:
تُعرَف البيئة لغةً بأنّها مجموعة العناصر الاصطناعية والطبيعية التي تُحِيطُ بالكائنات الحية (الإنسان والحيوان والنبات)، وهي أيضاً المنزل والمكان الذي يسكن فيه الكائن الحي، وبشكل عام هي جميع الظروف الخارجية التي تؤثر في شيء معين. [١]

Exemple après:
تعرف البيئة لغة بأنها مجموعة العناصر الاصطناعية والطبيعية التي تحيط بالكائنات الحية (الإنسان والحيوان والنبات)، وهي أيضا المنزل والمكان الذي يسكن فيه الكائن الحي، وبشكل عام هي جميع الظروف الخارجية التي تؤثر في شيء معين.


In [10]:
def get_label(score):

    if score <= 3:
        return 0

    elif score <= 7:
        return 1

    else:
        return 2

In [11]:
data = []

for text in cleaned_texts:
    data.append({
        "text": text,
        "score": 5,
        "label": 1,
        "source": "Mawdoo3"
    })

df_mawdoo3 = pd.DataFrame(data)

print(df_mawdoo3.shape)
print(df_mawdoo3["label"].value_counts())

print(df.head())
print("\nDataset shape:", df.shape)

(10, 4)
label
1    10
Name: count, dtype: int64
Empty DataFrame
Columns: [text, score, label, source]
Index: []

Dataset shape: (0, 4)


In [12]:
print("Dataset information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df["label"].value_counts())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    0 non-null      object
 1   score   0 non-null      object
 2   label   0 non-null      object
 3   source  0 non-null      object
dtypes: object(4)
memory usage: 132.0+ bytes
None

Missing values:
text      0
score     0
label     0
source    0
dtype: int64

Class distribution:
Series([], Name: count, dtype: int64)


In [13]:
pd.set_option("display.max_colwidth", 150)

for index, row in df.iterrows():
    print(f"\nObservation {index}")
    print("Texte :", row["text"])
    print("Score actuel :", row["score"])
    print("Classe actuelle :", row["label"])

In [14]:
url_gecc = "https://www.aljazeera.net/tag/climate/pollution/"

response_gecc = requests.get(
    url_gecc,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status:", response_gecc.status_code)
print("Content-Type:", response_gecc.headers.get("Content-Type"))
print("HTML length:", len(response_gecc.text))

Status: 200
Content-Type: text/html; charset=utf-8
HTML length: 333351


In [15]:
print(response_gecc.text)

<!DOCTYPE html>
<html lang="ar" dir="rtl" class="theme-aja">
<head>
	<meta charset="utf-8" />
	<meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no" />
	<meta http-equiv="content-type" content="text/html; charset=UTF-8" />
	<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
	<link rel="shortcut icon" href="/favicon_aja-v2.ico">
	<title data-rh="true">التلوث البيئي | الجزيرة نت</title><meta data-rh="true" name="robots" content="max-image-preview:large"/><meta data-rh="true" name="title" content="التلوث البيئي | الجزيرة نت"/><meta data-rh="true" name="description" content="ابقَ على اطلاعٍ مع أحدث التحديثات حول تلوث من قبل الجزيرة. استكشف المقالات العميقة والتحليلات وأخبار اللحظة، برعاية فريقنا المكرس من الصحفيين."/><link data-rh="true" rel="canonical" href="https://www.aljazeera.net/tag/climate/pollution/"/><script data-rh="true" type="application/ld+json">{"@context":"https://schema.org/","@type":"CollectionPage","headline":"التلوث البيئي | ال

In [16]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url_aljazeera = "https://www.aljazeera.net/tag/climate/pollution/"

response_aljazeera = requests.get(
    url_aljazeera,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

soup_aljazeera = BeautifulSoup(response_aljazeera.text, "html.parser")

links = []

for a in soup_aljazeera.find_all("a", href=True):
    title = a.get_text(" ", strip=True)
    link = urljoin(url_aljazeera, a["href"])

    if title and "aljazeera.net" in link:
        if link not in [item["url"] for item in links]:
            links.append({
                "title": title,
                "url": link
            })

print("Nombre de liens :", len(links))

for item in links[:20]:
    print(item["title"])
    print(item["url"])
    print("---")

Nombre de liens : 54
تخطى إلى المحتوى الرئيسي
https://www.aljazeera.net/tag/climate/pollution/#main-content
---
البث الحي
https://www.aljazeera.net/video/live
---
الرئيسية
https://www.aljazeera.net/
---
أخبار
https://www.aljazeera.net/news/
---
اقتصاد
https://www.aljazeera.net/ebusiness
---
رياضة
https://www.aljazeera.net/sport
---
رأي
https://www.aljazeera.net/opinions
---
ميدان
https://www.aljazeera.net/dimensions
---
متخصصة
https://www.aljazeera.net/verticals
---
محليات
https://www.aljazeera.net/local
---
فيديو
https://www.aljazeera.net/videos
---
الحساب
https://www.aljazeera.net/account
---
فلسطين
https://www.aljazeera.net/where/mideast/arab/palestine/
---
إيران
https://www.aljazeera.net/where/mideast/iran/
---
اليمن
https://www.aljazeera.net/where/mideast/arab/yemen/
---
سوريا
https://www.aljazeera.net/where/mideast/arab/syria/
---
لبنان
https://www.aljazeera.net/where/mideast/arab/lebanon/
---
أوكرانيا
https://www.aljazeera.net/where/intl/europe/ukraine/
---
اتحاد بلديات غزة: أزم

In [17]:
url_who = "https://www.who.int/ar"

response_who = requests.get(
    url_who,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status:", response_who.status_code)
print("HTML length:", len(response_who.text))

Status: 200
HTML length: 107040


In [18]:
article_links = []

for a in soup_aljazeera.find_all("a", href=True):
    title = a.get_text(" ", strip=True)
    link = urljoin(url_aljazeera, a["href"])

    if (
        title
        and link.startswith("https://www.aljazeera.net/news/")
        and link not in article_links
    ):
        article_links.append(link)

print("Nombre d'articles :", len(article_links))

for link in article_links:
    print(link)

Nombre d'articles : 6
https://www.aljazeera.net/news/
https://www.aljazeera.net/news/2026/8/18/gaza-municipalities-water-sewage-collapse-warning
https://www.aljazeera.net/news/2026/8/12/%d9%81%d9%8a-%d9%85%d9%86%d8%b7%d9%82%d8%a9-%d8%ad%d8%b3%d8%a7%d8%b3%d8%a9-%d8%a8%d9%8a%d8%a6%d9%8a%d8%a7-%d9%83%d9%8a%d9%81-%d9%8a%d8%a8%d8%af%d9%88-%d8%a7%d9%84%d8%aa%d8%b3%d8%b1%d8%a8
https://www.aljazeera.net/news/2026/7/2/%d9%83%d9%8a%d9%81-%d8%aa%d8%a8%d8%af%d9%88-%d8%ba%d8%b2%d8%a9-%d9%85%d9%86-%d8%a7%d9%84%d9%81%d8%b6%d8%a7%d8%a1-%d8%b5%d9%88%d8%b1-%d8%aa%d9%88%d8%ab%d9%82-%d8%ad%d8%ac%d9%85
https://www.aljazeera.net/news/2026/6/17/%d8%ae%d8%b1%d9%8a%d8%b7%d8%a9-%d9%84%d9%84%d8%ac%d8%b2%d9%8a%d8%b1%d8%a9-%d8%b6%d8%b1%d8%a8%d8%a7%d8%aa-%d8%a3%d9%88%d9%83%d8%b1%d8%a7%d9%86%d9%8a%d8%a9-%d8%aa%d8%b6%d9%8a%d9%82
https://www.aljazeera.net/news/2026/2/7/%d9%88%d8%a7%d8%b4%d9%86%d8%b7%d9%86-%d8%aa%d8%a8%d8%ad%d8%ab-%d8%b9%d9%86-%d9%85%d8%aa%d8%b7%d9%88%d8%b9%d9%8a%d9%86-%d9%84%d8%a7%d8%b3%d8%aa%d8%b6%d8

In [19]:
for link in article_links[:15]:
    print("\nURL :", link)

    try:
        r = requests.get(
            link,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=20
        )

        article_soup = BeautifulSoup(r.text, "html.parser")

        title = article_soup.find("h1")

        if title:
            print("Titre :", title.get_text(" ", strip=True))
        else:
            print("Titre non trouvé")

    except requests.RequestException as e:
        print("Erreur :", e)


URL : https://www.aljazeera.net/news/
Titre : أخبار العالم

URL : https://www.aljazeera.net/news/2026/8/18/gaza-municipalities-water-sewage-collapse-warning
Titre : 40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه

URL : https://www.aljazeera.net/news/2026/8/12/%d9%81%d9%8a-%d9%85%d9%86%d8%b7%d9%82%d8%a9-%d8%ad%d8%b3%d8%a7%d8%b3%d8%a9-%d8%a8%d9%8a%d8%a6%d9%8a%d8%a7-%d9%83%d9%8a%d9%81-%d9%8a%d8%a8%d8%af%d9%88-%d8%a7%d9%84%d8%aa%d8%b3%d8%b1%d8%a8
Titre : كيف يبدو التسرب النفطي من ناقلة جانحة قبالة سواحل ظفار العُمانية؟

URL : https://www.aljazeera.net/news/2026/7/2/%d9%83%d9%8a%d9%81-%d8%aa%d8%a8%d8%af%d9%88-%d8%ba%d8%b2%d8%a9-%d9%85%d9%86-%d8%a7%d9%84%d9%81%d8%b6%d8%a7%d8%a1-%d8%b5%d9%88%d8%b1-%d8%aa%d9%88%d8%ab%d9%82-%d8%ad%d8%ac%d9%85
Titre : كيف تبدو غزة من الفضاء؟ صور توثق حجم الدمار بعد 1000 يوم من حرب الإبادة

URL : https://www.aljazeera.net/news/2026/6/17/%d8%ae%d8%b1%d9%8a%d8%b7%d8%a9-%d9%84%d9%84%d8%ac%d8%b2%d9%8a%d8%b1%d8%a9-%d8%b6%d8%b1%d8%a8%d8%a7%d8%aa-%

In [20]:
import re

article_links = []

for a in soup_aljazeera.find_all("a", href=True):
    title = a.get_text(" ", strip=True)
    link = urljoin(url_aljazeera, a["href"])

    is_article = re.search(
        r"/news/\d{4}/\d{1,2}/\d{1,2}/",
        link
    )

    if title and is_article and link not in article_links:
        article_links.append(link)

print("Nombre d'articles :", len(article_links))

Nombre d'articles : 5


In [21]:
def extract_article(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    if response.status_code != 200:
        print("Erreur HTTP :", response.status_code)
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.get_text(" ", strip=True) if title_tag else ""

    paragraphs = []

    for p in soup.find_all("p"):
        text = p.get_text(" ", strip=True)

        if len(text) > 50:
            paragraphs.append(text)

    return {
        "title": title,
        "url": url,
        "paragraphs": paragraphs
    }

In [22]:
for url in article_links[:2]:
    article = extract_article(url)

    if article:
        print("\nTITRE :", article["title"])
        print("Nombre de paragraphes :", len(article["paragraphs"]))

        for p in article["paragraphs"][:3]:
            print("-", p)


TITRE : 40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه
Nombre de paragraphes : 14
- حذر اتحاد بلديات قطاع غزة من انهيار وشيك في منظومة المياه والصرف الصحي، بسبب شح الزيوت والوقود اللازمين لتشغيل المولدات والآبار ومحطات الضخ، بما يهدد السكان بالعطش ويزيد المخاطر الصحية والبيئية في القطاع.
- وقال رئيس بلدية خان يونس الدكتور علاء الدين البطة -متحدثا باسم الاتحاد- إن الزيوت أُدخلت إلى 40 مولدا فقط من أصل 700 مولد بالقطاع، مشيرا إلى أن منع إدخال المستلزمات اللازمة لتشغيل مولدات الآبار ومحطات الضخ والصرف الصحي أثر مباشرة في هذه المرافق للشهر الخامس على التوالي.
- وحذر البطة من أن الأزمة قد تقود إلى عطش جماعي لسكان القطاع وتؤثر سلبا في المرافق الحيوية، كما قد تؤدي إلى طفح مياه الصرف الصحي في الشوارع ووصولها إلى مخيمات النازحين، بما ينذر بانتشار مزيد من الأمراض والأوبئة.

TITRE : كيف يبدو التسرب النفطي من ناقلة جانحة قبالة سواحل ظفار العُمانية؟
Nombre de paragraphes : 20
- صورة أرشيفية لناقلة النفط "كارولين بيزنجي" الخاضعة للعقوبات (مارين ترافيك)
- وصل النفط المتسرب من ال

In [23]:
def clean_arabic_text(text):
    text = re.sub(r'\[[\d٠-٩]+\]', '', text)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    text = text.replace('ـ', '')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def filter_paragraphs(paragraphs):
    cleaned = []

    for p in paragraphs:
        text = clean_arabic_text(p)

        if len(text) >= 80:
            cleaned.append(text)

    return cleaned

In [24]:
def extract_article(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.get_text(" ", strip=True) if title_tag else ""

    paragraphs = [
        p.get_text(" ", strip=True)
        for p in soup.find_all("p")
    ]

    paragraphs = filter_paragraphs(paragraphs)

    return {
        "title": title,
        "url": url,
        "paragraphs": paragraphs
    }

In [25]:
all_articles = []

for article_id, url in enumerate(article_links, start=1):
    article = extract_article(url)

    if article and article["paragraphs"]:
        all_articles.append({
            "article_id": article_id,
            "source": "AlJazeera",
            "title": article["title"],
            "url": url,
            "paragraphs": article["paragraphs"]
        })

print("Articles récupérés :", len(all_articles))
print(
    "Paragraphes récupérés :",
    sum(len(a["paragraphs"]) for a in all_articles)
)

Articles récupérés : 5
Paragraphes récupérés : 110


In [26]:
for article in all_articles:
    print(f"\nArticle {article['article_id']}")
    print("Titre :", article["title"])
    print("Nombre de paragraphes :", len(article["paragraphs"]))


Article 1
Titre : 40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه
Nombre de paragraphes : 14

Article 2
Titre : كيف يبدو التسرب النفطي من ناقلة جانحة قبالة سواحل ظفار العُمانية؟
Nombre de paragraphes : 19

Article 3
Titre : كيف تبدو غزة من الفضاء؟ صور توثق حجم الدمار بعد 1000 يوم من حرب الإبادة
Nombre de paragraphes : 30

Article 4
Titre : خريطة للجزيرة.. ضربات أوكرانية تضيق الخناق على طرق الإمداد والطاقة في روسيا
Nombre de paragraphes : 33

Article 5
Titre : واشنطن تبحث عن "متطوعين" لاستضافة نفاياتها النووية إلى الأبد
Nombre de paragraphes : 14


In [27]:
for article in all_articles:
    if article["article_id"] in [3, 4]:
        print("\n" + "=" * 100)
        print("ARTICLE :", article["title"])

        for i, paragraph in enumerate(article["paragraphs"], start=1):
            print(f"\n[{i}] {paragraph}")


ARTICLE : كيف تبدو غزة من الفضاء؟ صور توثق حجم الدمار بعد 1000 يوم من حرب الإبادة

[1] خيام وملاجئ مؤقتة تؤوي نازحين فلسطينيين وسط أنقاض المباني قرب مخيم الشاطئ غربي مدينة غزة (الفرنسية)

[2] تجاوز قطاع غزة حاجز الألف يوم على اندلاع حرب الإبادة، وسط مؤشرات إنسانية واقتصادية توصف بالأخطر في التاريخ الحديث، ولم تعد الأزمة تقتصر على الحصيلة التراكمية للضحايا، بل امتدت لتحدث صدمة تنموية غير مسبوقة.

[3] وفي هذا السياق، حللت وحدة المصادر المفتوحة في شبكة الجزيرة صور أقمار صناعية وثقت التغيرات الهيكلية الكبيرة التي شهدها القطاع خلال ألف يوم من العدوان، حيث ترصد الصور وضع غزة قبل العملية البرية التي شنها جيش الاحتلال الإسرائيلي في 27 أكتوبر/تشرين الأول 2023، وصولا إلى التحول الكبير الذي وثقته أحدث اللقطات الفضائية في يونيو/حزيران الماضي.

[4] وتظهر صور الأقمار الصناعية تحول قطاع غزة، من بيت حانون شمالا وصولا إلى مدينة رفح جنوبا، إلى منطقة مدمرة بالكامل، إثر حرب تسببت في تضرر أو تدمير أغلبية المباني والمرافق الحيوية، وسط دمار فعلي طال 83% من المنشآت، وتضرر 94% من المؤسسات الصحية.

[5] تكشف ال

In [28]:
import pandas as pd

rows = []

for article in all_articles:
    for i, paragraph in enumerate(article["paragraphs"], start=1):
        rows.append({
            "article_id": article["article_id"],
            "paragraph_id": i,
            "title": article["title"],
            "text": paragraph,
            "score": None,
            "label": None,
            "source": article["source"]
        })

df = pd.DataFrame(rows)

print("Dimensions :", df.shape)
display(df.head())

Dimensions : (110, 7)


,article_id,paragraph_id,title,text,score,label,source
0,1,1,40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه,حذر اتحاد بلديات قطاع غزة من انهيار وشيك في منظومة المياه والصرف الصحي، بسبب شح الزيوت والوقود اللازمين لتشغيل المولدات والآبار ومحطات الضخ، بما ي...,None,None,AlJazeera
1,1,2,40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه,وقال رئيس بلدية خان يونس الدكتور علاء الدين البطة -متحدثا باسم الاتحاد- إن الزيوت أدخلت إلى 40 مولدا فقط من أصل 700 مولد بالقطاع، مشيرا إلى أن منع...,None,None,AlJazeera
2,1,3,40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه,وحذر البطة من أن الأزمة قد تقود إلى عطش جماعي لسكان القطاع وتؤثر سلبا في المرافق الحيوية، كما قد تؤدي إلى طفح مياه الصرف الصحي في الشوارع ووصولها ...,None,None,AlJazeera
3,1,4,40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه,وأكد الاتحاد أن البلديات لم تتلق -منذ مطلع مارس/آذار الماضي- أي كميات من الوقود المخصص لتشغيل آليات إزالة الركام، محذرا من توقف خدمات بلدية أساسية...,None,None,AlJazeera
4,1,5,40 مولدا من أصل 700.. بلديات غزة تحذر من انهيار وشيك لمنظومة المياه,ولا يقتصر أثر نقص الوقود والزيوت على مرافق المياه والخدمات البلدية، إذ أكد مدير الإغاثة الطبية في قطاع غزة الدكتور محمد أبو عفش أن القطاع الصحي يو...,None,None,AlJazeera


In [29]:
print("Nombre de doublons :", df["text"].duplicated().sum())

print("\nLongueur des paragraphes :")
print(df["text"].str.len().describe())

print("\nParagraphes les plus courts :")
display(
    df.loc[df["text"].str.len().sort_values().index]
      [["article_id", "text"]]
      .head(10)
)

Nombre de doublons : 0

Longueur des paragraphes :
count    110.000000
mean     227.009091
std       72.663924
min       90.000000
25%      181.250000
50%      217.500000
75%      270.500000
max      510.000000
Name: text, dtype: float64

Paragraphes les plus courts :


,article_id,text
63,4,حريق في منشأة داخل مصفاة نفط بموسكو عقب هجوم أوكراني بمسيرة، 16 يونيو/حزيران 2026 (رويترز)
43,3,وتكشف الصور عن تكدس شديد للخيام وتلاصقها داخل تجمعات عشوائية، مع ندرة المساحات الفاصلة بينها.
33,3,خيام وملاجئ مؤقتة تؤوي نازحين فلسطينيين وسط أنقاض المباني قرب مخيم الشاطئ غربي مدينة غزة (الفرنسية)
66,4,وركزت هذه الموجة من الهجمات على شل محطات ومصافي النفط، واستهداف المعابر والجسور الحيوية في العمق الروسي.
90,4,على جبهة بحر آزوف، طال الدمار منشآت ومخازن عسكرية في خليج ستريليتسكا بسيفاستوبول، وميناء تيمريوك، وميناء كافكاز.
96,5,"لافتات ""ممنوع الدخول"" تحيط بمنجم يورانيوم سابق في مدينة جيفري بولاية وايومنغ الأمريكية حيث دفنت النفايات المشعة (غيتي)"
64,4,شهد النصف الأول من يونيو/حزيران الجاري تصعيدا ملحوظا في الهجمات الأوكرانية التي استهدفت العمق الروسي بالطائرات المسيرة.
45,3,وتزداد هذه المعاناة لدى فئة ذوي الإعاقة، حيث يواجه 61% منهم صعوبات حادة في الحركة نتيجة الممرات الطينية والافتقار للمرافق المهيأة.
76,4,وبذلك أصبحت الممرات البرية الممتدة بين ماريوبول ودونيتسك وبيرديانسك جزءا من خريطة الضغط الأوكراني على طرق الإمداد الروسية في الجنوب.
84,4,وامتدت الضربات كذلك إلى خزان وقود في منطقة ميليروفو بمقاطعة روستوف، ومحطة نفط كويبيشيف في سامارا، خلال الفترة بين 9 و14 يونيو/حزيران.


In [30]:
def get_label(score):
    if score <= 3:
        return 0
    elif score <= 7:
        return 1
    else:
        return 2

In [31]:
import re

def clean_arabic_text(text):
    # Supprimer les diacritiques arabes
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)

    # Supprimer le Tatweel
    text = text.replace('ـ', '')

    # Normaliser les espaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


df["text"] = df["text"].apply(clean_arabic_text)

# Supprimer les doublons
df = df.drop_duplicates(subset=["text"])

# Supprimer les textes trop courts
df = df[df["text"].str.len() >= 80]

df = df.reset_index(drop=True)

print("Nombre de paragraphes après nettoyage :", len(df))


Nombre de paragraphes après nettoyage : 110


In [32]:
for i, row in df.iterrows():
    print(f"\nID : {i}")
    print(f"Article : {row['article_id']}")
    print(row["text"])


ID : 0
Article : 1
حذر اتحاد بلديات قطاع غزة من انهيار وشيك في منظومة المياه والصرف الصحي، بسبب شح الزيوت والوقود اللازمين لتشغيل المولدات والآبار ومحطات الضخ، بما يهدد السكان بالعطش ويزيد المخاطر الصحية والبيئية في القطاع.

ID : 1
Article : 1
وقال رئيس بلدية خان يونس الدكتور علاء الدين البطة -متحدثا باسم الاتحاد- إن الزيوت أدخلت إلى 40 مولدا فقط من أصل 700 مولد بالقطاع، مشيرا إلى أن منع إدخال المستلزمات اللازمة لتشغيل مولدات الآبار ومحطات الضخ والصرف الصحي أثر مباشرة في هذه المرافق للشهر الخامس على التوالي.

ID : 2
Article : 1
وحذر البطة من أن الأزمة قد تقود إلى عطش جماعي لسكان القطاع وتؤثر سلبا في المرافق الحيوية، كما قد تؤدي إلى طفح مياه الصرف الصحي في الشوارع ووصولها إلى مخيمات النازحين، بما ينذر بانتشار مزيد من الأمراض والأوبئة.

ID : 3
Article : 1
وأكد الاتحاد أن البلديات لم تتلق -منذ مطلع مارس/آذار الماضي- أي كميات من الوقود المخصص لتشغيل آليات إزالة الركام، محذرا من توقف خدمات بلدية أساسية تشمل فتح الطرق وجمع النفايات، إضافة إلى خدمات المياه والصرف الصحي.

ID : 4
Article : 1
و

In [33]:
df = df[~df.index.isin([33, 63, 96])].copy()
df = df.reset_index(drop=True)

print("Nombre de paragraphes apres elimination de qlqs paragraphes :", len(df))

Nombre de paragraphes apres elimination de qlqs paragraphes : 107


In [34]:
# IDs du DataFrame AVANT la suppression des légendes
class_2_ids = [
    0, 1, 2, 3, 10,
    14, 15, 17, 18, 20, 22, 23, 24, 25, 32,
    41, 53, 54, 55, 56, 57, 58,
    97, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109
]

class_1_ids = [
    16, 19, 27, 28,
    42, 44, 60
]

# Initialisation
df["score"] = 1
df["label"] = 0

# Annotation des classes 1 et 2
df.loc[df.index.isin(class_1_ids), "score"] = 5
df.loc[df.index.isin(class_1_ids), "label"] = 1

df.loc[df.index.isin(class_2_ids), "score"] = 9
df.loc[df.index.isin(class_2_ids), "label"] = 2

print(df["label"].value_counts().sort_index())
print(df[["text", "score", "label"]].head(10))

label
0    69
1     7
2    31
Name: count, dtype: int64
                                                                                                                                                    text  \
0  حذر اتحاد بلديات قطاع غزة من انهيار وشيك في منظومة المياه والصرف الصحي، بسبب شح الزيوت والوقود اللازمين لتشغيل المولدات والآبار ومحطات الضخ، بما ي...   
1  وقال رئيس بلدية خان يونس الدكتور علاء الدين البطة -متحدثا باسم الاتحاد- إن الزيوت أدخلت إلى 40 مولدا فقط من أصل 700 مولد بالقطاع، مشيرا إلى أن منع...   
2  وحذر البطة من أن الأزمة قد تقود إلى عطش جماعي لسكان القطاع وتؤثر سلبا في المرافق الحيوية، كما قد تؤدي إلى طفح مياه الصرف الصحي في الشوارع ووصولها ...   
3  وأكد الاتحاد أن البلديات لم تتلق -منذ مطلع مارس/آذار الماضي- أي كميات من الوقود المخصص لتشغيل آليات إزالة الركام، محذرا من توقف خدمات بلدية أساسية...   
4  ولا يقتصر أثر نقص الوقود والزيوت على مرافق المياه والخدمات البلدية، إذ أكد مدير الإغاثة الطبية في قطاع غزة الدكتور محمد أبو عفش أن القطاع الصحي يو...   
5  وأضاف

In [35]:
print(df.columns.tolist())
print(df[["article_id", "paragraph_id"]].head(15))

['article_id', 'paragraph_id', 'title', 'text', 'score', 'label', 'source']
    article_id  paragraph_id
0            1             1
1            1             2
2            1             3
3            1             4
4            1             5
5            1             6
6            1             7
7            1             8
8            1             9
9            1            10
10           1            11
11           1            12
12           1            13
13           1            14
14           2             1


In [36]:
offsets = {
    1: -1,
    2: 13,
    3: 32,
    4: 62,
    5: 95
}

df["original_id"] = (
    df["article_id"].map(offsets) + df["paragraph_id"]
)

In [37]:
df["score"] = 1
df["label"] = 0

df.loc[df["original_id"].isin(class_1_ids), ["score", "label"]] = [5, 1]

df.loc[df["original_id"].isin(class_2_ids), ["score", "label"]] = [9, 2]

In [38]:
print("Répartition des labels :")
print(df["label"].value_counts().sort_index())

print("\nVérification des scores :")
print(df.groupby("label")["score"].unique())

Répartition des labels :
label
0    66
1     7
2    34
Name: count, dtype: int64

Vérification des scores :
label
0    [1]
1    [5]
2    [9]
Name: score, dtype: object


In [39]:
df.to_csv("environment_arabic_annotated.csv",
          index=False,
          encoding="utf-8-sig")

In [40]:
df_aljazeera = pd.read_csv(
    "environment_arabic_annotated.csv"
)

In [41]:
df_final = pd.concat(
    [df_aljazeera, df_mawdoo3],
    ignore_index=True
)

print("Dataset final :", df_final.shape)
print("\nRépartition :")
print(df_final["label"].value_counts().sort_index())

Dataset final : (117, 8)

Répartition :
label
0    66
1    17
2    34
Name: count, dtype: int64


In [42]:
print("Dimensions :", df_final.shape)

print("\nValeurs manquantes :")
print(df_final.isnull().sum())

print("\nDoublons :")
print(df_final.duplicated(subset=["text"]).sum())

print("\nRépartition par source :")
print(df_final["source"].value_counts())

Dimensions : (117, 8)

Valeurs manquantes :
article_id      10
paragraph_id    10
title           10
text             0
score            0
label            0
source           0
original_id     10
dtype: int64

Doublons :
0

Répartition par source :
source
AlJazeera    107
Mawdoo3       10
Name: count, dtype: int64


In [43]:
#normalisation
import re

def normalize_arabic(text):

    # Supprimer les signes de vocalisation
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)

    # Supprimer le Tatweel
    text = text.replace('ـ', '')

    # Uniformiser Alif
    text = re.sub('[إأآ]', 'ا', text)

    # Uniformiser Alif Maqsura
    text = text.replace('ى', 'ي')

    # Uniformiser espaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

df_final["text_normalized"] = df_final["text"].apply(normalize_arabic)

In [44]:
!pip install -q nltk 

In [45]:
#tokenization
import nltk
from nltk.tokenize import wordpunct_tokenize
df_final["tokens"] = df_final["text_normalized"].apply(
    wordpunct_tokenize
)

print(df_final[["text_normalized", "tokens"]].head(3))

                                                                                                                                         text_normalized  \
0  حذر اتحاد بلديات قطاع غزة من انهيار وشيك في منظومة المياه والصرف الصحي، بسبب شح الزيوت والوقود اللازمين لتشغيل المولدات والابار ومحطات الضخ، بما ي...   
1  وقال رئيس بلدية خان يونس الدكتور علاء الدين البطة -متحدثا باسم الاتحاد- ان الزيوت ادخلت الي 40 مولدا فقط من اصل 700 مولد بالقطاع، مشيرا الي ان منع...   
2  وحذر البطة من ان الازمة قد تقود الي عطش جماعي لسكان القطاع وتؤثر سلبا في المرافق الحيوية، كما قد تؤدي الي طفح مياه الصرف الصحي في الشوارع ووصولها ...   

                                                                                                                                                  tokens  
0  [حذر, اتحاد, بلديات, قطاع, غزة, من, انهيار, وشيك, في, منظومة, المياه, والصرف, الصحي, ،, بسبب, شح, الزيوت, والوقود, اللازمين, لتشغيل, المولدات, وال...  
1  [وقال, رئيس, بلدية, خان, يونس, الدكتور, علاء, الدين, البطة, -,

In [46]:
print(df_final.columns)

Index(['article_id', 'paragraph_id', 'title', 'text', 'score', 'label',
       'source', 'original_id', 'text_normalized', 'tokens'],
      dtype='object')


In [47]:
df_final["token_count"] = df_final["tokens"].apply(len)

print(df_final["token_count"].describe())

count    117.000000
mean      43.666667
std       15.267726
min       10.000000
25%       35.000000
50%       42.000000
75%       51.000000
max       99.000000
Name: token_count, dtype: float64


In [48]:
#installer stop words arabe
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

arabic_stopwords = set(stopwords.words('arabic'))
important_words = {"لا", "لم", "لن", "ليس", "ما"}
arabic_stopwords = arabic_stopwords - important_words
print("Nombre de stop words :", len(arabic_stopwords))
print(list(arabic_stopwords)[:20])

Nombre de stop words : 696
['خ', 'آها', 'مايو', 'لوما', 'فيم', 'إنَّ', 'ذ', 'ألا', 'والذي', 'أخبر', 'ما انفك', 'ثلاثمائة', 'تسعين', 'ة', 'سبحان', 'عوض', 'تينك', 'بلى', 'ميم', 'فيفري']


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [49]:
def remove_stopwords(tokens):
    return [
        word for word in tokens
        if word not in arabic_stopwords
    ]

df_final["tokens_no_stopwords"] = df_final["tokens"].apply(
    remove_stopwords
)
arabic_stopwords = arabic_stopwords - important_words

df_final["tokens_no_stopwords"] = df_final["tokens"].apply(
    lambda tokens: [w for w in tokens if w not in arabic_stopwords]
)

In [50]:
index = 0

print("Avant :", df_final.loc[index, "tokens"])
print("\nAprès :", df_final.loc[index, "tokens_no_stopwords"])
df_final["token_count_after"] = (
    df_final["tokens_no_stopwords"].apply(len)
)

print(df_final["token_count_after"].describe())

Avant : ['حذر', 'اتحاد', 'بلديات', 'قطاع', 'غزة', 'من', 'انهيار', 'وشيك', 'في', 'منظومة', 'المياه', 'والصرف', 'الصحي', '،', 'بسبب', 'شح', 'الزيوت', 'والوقود', 'اللازمين', 'لتشغيل', 'المولدات', 'والابار', 'ومحطات', 'الضخ', '،', 'بما', 'يهدد', 'السكان', 'بالعطش', 'ويزيد', 'المخاطر', 'الصحية', 'والبيئية', 'في', 'القطاع', '.']

Après : ['حذر', 'اتحاد', 'بلديات', 'قطاع', 'غزة', 'انهيار', 'وشيك', 'منظومة', 'المياه', 'والصرف', 'الصحي', '،', 'بسبب', 'شح', 'الزيوت', 'والوقود', 'اللازمين', 'لتشغيل', 'المولدات', 'والابار', 'ومحطات', 'الضخ', '،', 'يهدد', 'السكان', 'بالعطش', 'ويزيد', 'المخاطر', 'الصحية', 'والبيئية', 'القطاع', '.']
count    117.000000
mean      37.803419
std       12.996179
min        9.000000
25%       31.000000
50%       35.000000
75%       46.000000
max       85.000000
Name: token_count_after, dtype: float64


In [51]:
#Stemming (racinisation)
from nltk.stem.isri import ISRIStemmer

stemmer = ISRIStemmer()

words = ["المياه", "البيئية", "التلوث", "المولدات"]

for word in words:
    print(word, "→", stemmer.stem(word))

المياه → ياه
البيئية → بيئ
التلوث → تلث
المولدات → ولد


In [52]:
from nltk.stem.isri import ISRIStemmer

stemmer = ISRIStemmer()

df_final["tokens_stemmed"] = df_final["tokens_no_stopwords"].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)

print("Original :", df_final["tokens_no_stopwords"].iloc[0])
print("\nStemmed :", df_final["tokens_stemmed"].iloc[0])

Original : ['حذر', 'اتحاد', 'بلديات', 'قطاع', 'غزة', 'انهيار', 'وشيك', 'منظومة', 'المياه', 'والصرف', 'الصحي', '،', 'بسبب', 'شح', 'الزيوت', 'والوقود', 'اللازمين', 'لتشغيل', 'المولدات', 'والابار', 'ومحطات', 'الضخ', '،', 'يهدد', 'السكان', 'بالعطش', 'ويزيد', 'المخاطر', 'الصحية', 'والبيئية', 'القطاع', '.']

Stemmed : ['حذر', 'تحد', 'بلد', 'قطع', 'غزة', 'هير', 'وشك', 'نظم', 'ياه', 'صرف', 'صحي', '،', 'سبب', 'شح', 'زيت', 'وقد', 'لزم', 'شغل', 'ولد', 'ابر', 'محط', 'لضخ', '،', 'هدد', 'سكن', 'عطش', 'يزد', 'خطر', 'صحة', 'بيئ', 'قطع', '.']


In [53]:
!pip install -q camel-tools

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [54]:
!camel_data -i light

The following packages will be installed: 'morphology-db-lev-01', 'disambig-mle-calima-msa-r13', 'dialectid-model26', 'morphology-db-msa-r13', 'disambig-mle-calima-egy-r13', 'morphology-db-glf-01', 'morphology-db-msa-s31', 'morphology-db-egy-r13'
Extracting package 'morphology-db-lev-01': 100%|█| 10.6M/10.6M [00:00<00:00, 1.1
Extracting package 'disambig-mle-calima-msa-r13': 100%|█| 88.7M/88.7M [00:00<00:
Extracting package 'dialectid-model26': 100%|█| 275M/275M [00:00<00:00, 1.08GB/s
Extracting package 'morphology-db-msa-r13': 100%|█| 40.5M/40.5M [00:00<00:00, 1.
Extracting package 'disambig-mle-calima-egy-r13': 100%|█| 27.2M/27.2M [00:00<00:
Extracting package 'morphology-db-glf-01': 100%|█| 7.98M/7.98M [00:00<00:00, 993
Extracting package 'morphology-db-msa-s31': 100%|█| 44.8M/44.8M [00:00<00:00, 1.
Extracting package 'morphology-db-egy-r13': 100%|█| 67.3M/67.3M [00:00<00:00, 1.


In [55]:
#lemmatisation
import camel_tools
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

db = MorphologyDB.builtin_db()
analyzer = Analyzer(db)

print("Analyseur morphologique chargé")

Analyseur morphologique chargé


In [56]:
def lemmatize_word(word):
    analyses = analyzer.analyze(word)

    if analyses:
        return analyses[0].get("lex", word)

    return word


def lemmatize_tokens(tokens):
    return [lemmatize_word(word) for word in tokens]


df_final["tokens_lemmatized"] = df_final["tokens_no_stopwords"].apply(
    lemmatize_tokens
)

In [57]:
index = 0

print("Texte original :")
print(df_final["text"].iloc[index])

print("\nAprès suppression des stop words :")
print(df_final["tokens_no_stopwords"].iloc[index])

print("\nAprès stemming :")
print(df_final["tokens_stemmed"].iloc[index])

print("\nAprès lemmatisation :")
print(df_final["tokens_lemmatized"].iloc[index])

Texte original :
حذر اتحاد بلديات قطاع غزة من انهيار وشيك في منظومة المياه والصرف الصحي، بسبب شح الزيوت والوقود اللازمين لتشغيل المولدات والآبار ومحطات الضخ، بما يهدد السكان بالعطش ويزيد المخاطر الصحية والبيئية في القطاع.

Après suppression des stop words :
['حذر', 'اتحاد', 'بلديات', 'قطاع', 'غزة', 'انهيار', 'وشيك', 'منظومة', 'المياه', 'والصرف', 'الصحي', '،', 'بسبب', 'شح', 'الزيوت', 'والوقود', 'اللازمين', 'لتشغيل', 'المولدات', 'والابار', 'ومحطات', 'الضخ', '،', 'يهدد', 'السكان', 'بالعطش', 'ويزيد', 'المخاطر', 'الصحية', 'والبيئية', 'القطاع', '.']

Après stemming :
['حذر', 'تحد', 'بلد', 'قطع', 'غزة', 'هير', 'وشك', 'نظم', 'ياه', 'صرف', 'صحي', '،', 'سبب', 'شح', 'زيت', 'وقد', 'لزم', 'شغل', 'ولد', 'ابر', 'محط', 'لضخ', '،', 'هدد', 'سكن', 'عطش', 'يزد', 'خطر', 'صحة', 'بيئ', 'قطع', '.']

Après lemmatisation :
['حِذْر', 'ٱِتِّحاد', 'بَلَدِيَّة', 'قاطِع', 'غَزَّة', 'ٱِنْهِيار', 'وَشْي', 'مَنْظُوم', 'ماء', 'صَرْف', 'صِحِّيّ', '،', 'سَبَب', 'شُحّ', 'زَيْت', 'وَقُود', 'لازِم', 'تَشْغِيل', 'مُوَلِّد', '

In [58]:
df_final["lemmatized_count"] = df_final["tokens_lemmatized"].apply(len)

print(df_final["lemmatized_count"].describe())

count    117.000000
mean      37.803419
std       12.996179
min        9.000000
25%       31.000000
50%       35.000000
75%       46.000000
max       85.000000
Name: lemmatized_count, dtype: float64


In [59]:
def remove_punctuation(tokens):
    return [
        word for word in tokens
        if re.search(r'\w', word, flags=re.UNICODE)
    ]

df_final["tokens_clean"] = df_final["tokens_no_stopwords"].apply(
    remove_punctuation
)

print(df_final["tokens_clean"].iloc[0])

['حذر', 'اتحاد', 'بلديات', 'قطاع', 'غزة', 'انهيار', 'وشيك', 'منظومة', 'المياه', 'والصرف', 'الصحي', 'بسبب', 'شح', 'الزيوت', 'والوقود', 'اللازمين', 'لتشغيل', 'المولدات', 'والابار', 'ومحطات', 'الضخ', 'يهدد', 'السكان', 'بالعطش', 'ويزيد', 'المخاطر', 'الصحية', 'والبيئية', 'القطاع']


In [60]:
print(pd.crosstab(df_final["score"], df_final["label"]))

print("\nRépartition des classes :")
print(df_final["label"].value_counts().sort_index())

label   0   1   2
score            
1      66   0   0
5       0  17   0
9       0   0  34

Répartition des classes :
label
0    66
1    17
2    34
Name: count, dtype: int64


In [61]:
df_final["text_lemmatized"] = df_final["tokens_lemmatized"].apply(
    lambda tokens: " ".join(tokens)
)

print(df_final[["text_lemmatized", "label"]].head())

                                                                                                                                         text_lemmatized  \
0  حِذْر ٱِتِّحاد بَلَدِيَّة قاطِع غَزَّة ٱِنْهِيار وَشْي مَنْظُوم ماء صَرْف صِحِّيّ ، سَبَب شُحّ زَيْت وَقُود لازِم تَشْغِيل مُوَلِّد بِئْر مَحَطَّة...   
1  قال رَئِيس بَلَد خان يُونِس دُكْتُور عَلاء أَلَدّ لَبَط - مُتَحَدِّث باسِم ٱِتِّحاد - أَنْ زَيْت أَدْخَل آل 40 مُوَلِّد فَقَط أَصَّل 700 مُوَلِّد ...   
2  حِذْر لَبَط أَنْ لازَم قاد آل عَطَش جِماع سَكّان قاطِع أَثَّر سَلْب مِرْفَق حَيَوِيَّة ، أَدّ آل طَفْح ماء صَرْف صِحِّيّ شارِع وُصُول آل مُخَيَّم ...   
3  واكِد ٱِتِّحاد أَنْ بَلَدِيَّة لَمّ تَلَقَّى - مَطْلَع / آذار ماضِي - أَيْ كَمِيّ وَقُود مُخَصَّص تَشْغِيل أَلْيَة أَزال رُكام ، مُحَذِّر تَوَقُّف...   
4  ٱِقْتَصَر أَثِر نَقَص وَقُود زَيْت عَلِيّ مِرْفَق ماء خِدْمَة بَلَدِيَّة ، إِذ أَكَّد مُدِير إِغاثَة طِبِّيّ قاطِع غَزَّة دُكْتُور مُحَمَّد آبِي ع...   

   label  
0      2  
1      2  
2      2  
3      2  
4      0

In [62]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(oov_token="<OOV>")

tokenizer.fit_on_texts(df_final["text_lemmatized"])

sequences = tokenizer.texts_to_sequences(
    df_final["text_lemmatized"]
)

print("Taille du vocabulaire :", len(tokenizer.word_index))
print("\nPremier texte :", df_final["text_lemmatized"].iloc[0])
print("\nSéquence numérique :", sequences[0])

Taille du vocabulaire : 1571

Premier texte : حِذْر ٱِتِّحاد بَلَدِيَّة قاطِع غَزَّة ٱِنْهِيار وَشْي مَنْظُوم ماء صَرْف صِحِّيّ ، سَبَب شُحّ زَيْت وَقُود لازِم تَشْغِيل مُوَلِّد بِئْر مَحَطَّة ضَخّ ، هَدَّد سَكّان عَطَش زاد مُخاطِر صِحِّيَّة بِيئِيّ قاطِع .

Séquence numérique : [266, 103, 104, 6, 11, 267, 674, 268, 8, 61, 37, 2, 189, 675, 75, 27, 143, 190, 191, 398, 17, 192, 2, 399, 144, 269, 400, 193, 62, 76, 6]


In [63]:
#Padding : uniformiser les séquences
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 85

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

y = df_final["label"].values

print("Dimension de X :", X.shape)
print("Dimension de y :", y.shape)
print("\nPremière séquence après padding :")
print(X[0])

Dimension de X : (117, 85)
Dimension de y : (117,)

Première séquence après padding :
[266 103 104   6  11 267 674 268   8  61  37   2 189 675  75  27 143 190
 191 398  17 192   2 399 144 269 400 193  62  76   6   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0]


In [64]:
from sklearn.model_selection import train_test_split

texts = df_final["text_lemmatized"].values
y = df_final["label"].values

# Première séparation : 80 % développement, 20 % test
X_dev, X_test_text, y_dev, y_test = train_test_split(
    texts,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Deuxième séparation : train et validation
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    random_state=42,
    stratify=y_dev
)

print("Train :", len(X_train_text))
print("Validation :", len(X_val_text))
print("Test :", len(X_test_text))

Train : 74
Validation : 19
Test : 24


In [65]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(oov_token="<OOV>")

# Apprentissage du vocabulaire sur Train uniquement
tokenizer.fit_on_texts(X_train_text)

# Transformation des trois ensembles
train_sequences = tokenizer.texts_to_sequences(X_train_text)
val_sequences = tokenizer.texts_to_sequences(X_val_text)
test_sequences = tokenizer.texts_to_sequences(X_test_text)

# Padding
max_length = 85

X_train = pad_sequences(train_sequences, maxlen=max_length, padding="post", truncating="post")
X_val = pad_sequences(val_sequences, maxlen=max_length, padding="post", truncating="post")
X_test = pad_sequences(test_sequences, maxlen=max_length, padding="post", truncating="post")

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulaire :", vocab_size)
print("X_train :", X_train.shape)
print("X_val :", X_val.shape)
print("X_test :", X_test.shape)

Vocabulaire : 1172
X_train : (74, 85)
X_val : (19, 85)
X_test : (24, 85)


In [66]:
import pandas as pd

print("Train :")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation :")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest :")
print(pd.Series(y_test).value_counts().sort_index())

Train :
0    41
1    11
2    22
Name: count, dtype: int64

Validation :
0    11
1     3
2     5
Name: count, dtype: int64

Test :
0    14
1     3
2     7
Name: count, dtype: int64


In [67]:
#modèle RNN
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

tf.keras.utils.set_random_seed(42)

model_rnn = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),

    SimpleRNN(64),

    Dropout(0.3),

    Dense(3, activation="softmax")
])

model_rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_rnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [68]:
history_rnn = model_rnn.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=8,
    validation_data=(X_val, y_val),
    verbose=1
)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 383ms/step - accuracy: 0.5541 - loss: 1.0435 - val_accuracy: 0.5789 - val_loss: 1.0233
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7162 - loss: 0.7390 - val_accuracy: 0.4211 - val_loss: 1.0844
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9865 - loss: 0.3723 - val_accuracy: 0.4211 - val_loss: 1.1725
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.1552 - val_accuracy: 0.3684 - val_loss: 1.4091
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0664 - val_accuracy: 0.3158 - val_loss: 1.5183
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0386 - val_accuracy: 0.3684 - val_loss: 1.5329
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0279 - val_accuracy: 0.2632 - val_loss: 1.5349
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0163 - val_accuracy: 0.3684 - 

In [69]:
model_rnn.build(input_shape=(None, 85))

model_rnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 85, 64)         │        75,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 250,379 (978.05 KB)

 Trainable params: 83,459 (326.01 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 166,920 (652.04 KB)

In [70]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

model_rnn_v2 = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=32,
        mask_zero=True
    ),

    SimpleRNN(16),

    Dropout(0.5),

    Dense(3, activation="softmax")
])

model_rnn_v2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_rnn_v2 = model_rnn_v2.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=8,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 288ms/step - accuracy: 0.3784 - loss: 1.0962 - val_accuracy: 0.4737 - val_loss: 1.0217
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7027 - loss: 0.8976 - val_accuracy: 0.5263 - val_loss: 0.9901
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9189 - loss: 0.7472 - val_accuracy: 0.6316 - val_loss: 0.9790
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8919 - loss: 0.6254 - val_accuracy: 0.5789 - val_loss: 0.9825
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.4910 - val_accuracy: 0.6316 - val_loss: 0.9839
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9730 - loss: 0.3871 - val_accuracy: 0.6316 - val_loss: 0.9777
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9865 - loss: 0.3188 - val_accuracy: 0.6842 - val_loss: 0.9937
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.2430 - val_accuracy: 0.6316 - 

In [71]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model_rnn_v2.predict(X_test)
y_pred = y_pred_prob.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    target_names=["Non pertinent", "Environnement général", "Pollution"],
    zero_division=0
))

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred, labels=[0, 1, 2]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step
                       precision    recall  f1-score   support

        Non pertinent       0.56      0.64      0.60        14
Environnement général       0.00      0.00      0.00         3
            Pollution       0.38      0.43      0.40         7

             accuracy                           0.50        24
            macro avg       0.31      0.36      0.33        24
         weighted avg       0.44      0.50      0.47        24

Matrice de confusion :
[[9 0 5]
 [3 0 0]
 [4 0 3]]


In [72]:
# Bidirectional RNN 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, Bidirectional, Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

model_birnn = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=32,
        mask_zero=True
    ),
    Bidirectional(SimpleRNN(16)),
    Dropout(0.5),
    Dense(3, activation="softmax")
])

model_birnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_birnn.build(input_shape=(None, max_length))
model_birnn.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 85, 32)         │        37,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,171 (153.01 KB)

 Trainable params: 39,171 (153.01 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, Bidirectional, Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

model_birnn = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=32,
        mask_zero=True
    ),
    Bidirectional(SimpleRNN(16)),
    Dropout(0.5),
    Dense(3, activation="softmax")
])

model_birnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_birnn.build(input_shape=(None, max_length))
model_birnn.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 85, 32)         │        37,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,171 (153.01 KB)

 Trainable params: 39,171 (153.01 KB)

 Non-trainable params: 0 (0.00 B)

In [74]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_birnn = model_birnn.fit(
    X_train, y_train,
    epochs=30,
    batch_size=8,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 349ms/step - accuracy: 0.3243 - loss: 1.1115 - val_accuracy: 0.4737 - val_loss: 1.0475
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7838 - loss: 0.8743 - val_accuracy: 0.4737 - val_loss: 1.0309
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9459 - loss: 0.6896 - val_accuracy: 0.4737 - val_loss: 1.0134
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9324 - loss: 0.5216 - val_accuracy: 0.5263 - val_loss: 1.0286
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9459 - loss: 0.4372 - val_accuracy: 0.4737 - val_loss: 1.0174
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9730 - loss: 0.3083 - val_accuracy: 0.4737 - val_loss: 0.9884
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 0.2353 - val_accuracy: 0.5263 - val_loss: 0.9627
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 0.2261 - val_accuracy: 0.4737 - 

In [75]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob_bi = model_birnn.predict(X_test)
y_pred_bi = y_pred_prob_bi.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_bi,
    labels=[0, 1, 2],
    target_names=[
        "Non pertinent",
        "Environnement général",
        "Pollution"
    ],
    zero_division=0
))

print(confusion_matrix(y_test, y_pred_bi, labels=[0, 1, 2]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 714ms/step
                       precision    recall  f1-score   support

        Non pertinent       0.67      0.71      0.69        14
Environnement général       0.00      0.00      0.00         3
            Pollution       0.33      0.29      0.31         7

             accuracy                           0.50        24
            macro avg       0.33      0.33      0.33        24
         weighted avg       0.49      0.50      0.49        24

[[10  2  2]
 [ 1  0  2]
 [ 4  1  2]]


In [76]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

model_gru = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=32,
        mask_zero=True
    ),
    GRU(16),
    Dropout(0.5),
    Dense(3, activation="softmax")
])

model_gru.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_gru.build(input_shape=(None, max_length))
model_gru.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 85, 32)         │        37,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 16)             │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,955 (156.07 KB)

 Trainable params: 39,955 (156.07 KB)

 Non-trainable params: 0 (0.00 B)

In [77]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_gru = model_gru.fit(
    X_train, y_train,
    epochs=30,
    batch_size=8,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.3649 - loss: 1.0955 - val_accuracy: 0.6316 - val_loss: 1.0756
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5270 - loss: 1.0718 - val_accuracy: 0.5789 - val_loss: 1.0593
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5541 - loss: 1.0521 - val_accuracy: 0.5789 - val_loss: 1.0430
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5676 - loss: 1.0080 - val_accuracy: 0.5789 - val_loss: 1.0223
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5541 - loss: 0.9811 - val_accuracy: 0.5789 - val_loss: 0.9989
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5541 - loss: 0.9490 - val_accuracy: 0.5789 - val_loss: 0.9710
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5541 - loss: 0.8872 - val_accuracy: 0.5789 - val_loss: 0.9367
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5676 - loss: 0.7868 - val_accuracy: 0.5789 - v

In [78]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob_gru = model_gru.predict(X_test)
y_pred_gru = y_pred_prob_gru.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_gru,
    labels=[0, 1, 2],
    target_names=[
        "Non pertinent",
        "Environnement général",
        "Pollution"
    ],
    zero_division=0
))

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred_gru, labels=[0, 1, 2]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
                       precision    recall  f1-score   support

        Non pertinent       0.76      0.93      0.84        14
Environnement général       0.00      0.00      0.00         3
            Pollution       0.57      0.57      0.57         7

             accuracy                           0.71        24
            macro avg       0.45      0.50      0.47        24
         weighted avg       0.61      0.71      0.66        24

Matrice de confusion :
[[13  0  1]
 [ 1  0  2]
 [ 3  0  4]]


In [79]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

model_lstm = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=32,
        mask_zero=True
    ),
    LSTM(16),
    Dropout(0.5),
    Dense(3, activation="softmax")
])

model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_lstm.build(input_shape=(None, max_length))
model_lstm.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 85, 32)         │        37,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,691 (158.95 KB)

 Trainable params: 40,691 (158.95 KB)

 Non-trainable params: 0 (0.00 B)

In [80]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=30,
    batch_size=8,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.4459 - loss: 1.0923 - val_accuracy: 0.6842 - val_loss: 1.0856
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5946 - loss: 1.0730 - val_accuracy: 0.5789 - val_loss: 1.0676
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5541 - loss: 1.0454 - val_accuracy: 0.5789 - val_loss: 1.0429
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6216 - loss: 0.9805 - val_accuracy: 0.5789 - val_loss: 0.9922
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5541 - loss: 0.9169 - val_accuracy: 0.5789 - val_loss: 0.9411
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5270 - loss: 0.8443 - val_accuracy: 0.5789 - val_loss: 0.9086
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6757 - loss: 0.7394 - val_accuracy: 0.6316 - val_loss: 0.8612
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7973 - loss: 0.6319 - val_accuracy: 0.6316 - v

In [81]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob_lstm = model_lstm.predict(X_test)
y_pred_lstm = y_pred_prob_lstm.argmax(axis=1)

print(classification_report(
    y_test,
    y_pred_lstm,
    labels=[0, 1, 2],
    target_names=[
        "Non pertinent",
        "Environnement général",
        "Pollution"
    ],
    zero_division=0
))

print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred_lstm, labels=[0, 1, 2]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
                       precision    recall  f1-score   support

        Non pertinent       0.72      0.93      0.81        14
Environnement général       0.40      0.67      0.50         3
            Pollution       1.00      0.14      0.25         7

             accuracy                           0.67        24
            macro avg       0.71      0.58      0.52        24
         weighted avg       0.76      0.67      0.61        24

Matrice de confusion :
[[13  1  0]
 [ 1  2  0]
 [ 4  2  1]]
